# Notebook de Training Manuel - MLPipeline

Ce notebook permet d'exécuter manuellement chaque étape du pipeline d'entraînement en utilisant la classe `MLPipeline`.

## Étapes du pipeline MLPipeline :
1. **Chargement des données** - step_1_load_data
2. **Validation des données** - step_2_validate_data
3. **Transformation des données** - step_3_transform_data
4. **Préparation des données** - step_3_prepare_data
5. **Entraînement du modèle** - step_4_train_model
6. **Évaluation du modèle** - step_5_evaluate_model
7. **Monitoring de la performance** - step_6_monitor_performance
8. **Logging MLflow** - step_7_log_with_mlflow
9. **Nettoyage du modèle** - step_8_cleanup_model
10. **Gestion des stages** - step_9_manage_model_stages

## 0. Initialisation et Configuration

In [25]:
import os
import logging
import warnings
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Charger les variables d'environnement
load_dotenv(find_dotenv(".env"), override=True)
load_dotenv(find_dotenv(".env.secrets"), override=True)

# Configurer le logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Ignorer certains warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

print("✅ Initialisation terminée")

✅ Initialisation terminée


In [26]:
# Definir l'environnement de test
os.environ['ENV'] = 'test'

# Définir l'environnement
env = os.getenv('ENV', 'dev')
print(f'📍 Environment: {env}')

# Définir le nom de la configuration (sans chemin, juste le nom)
#CONFIG_NAME = "consumption"
#print(f'📂 Config name: {CONFIG_NAME}')

📍 Environment: test


## 1. Initialisation du MLPipeline

In [27]:
from ml.utils.pipelines.training_pipeline import MLPipeline

# Initialiser le pipeline avec config_name
pipeline = MLPipeline()

print('✅ MLPipeline initialisé')
print(f'\n📊 Configuration:')
print(f"  Model name: {pipeline.model_name}")
print(f"  Model type: {pipeline.config.get('model', {}).get('model_type')}")
print(f"  Test size: {pipeline.config.get('model', {}).get('test_size')}")
print(f"  Random state: {pipeline.config.get('model', {}).get('random_state')}")

2026-06-08 00:58:57,908 - ml.utils.pipelines.training_pipeline - INFO - Pipeline MLOps initialisé avec model_name=consumption_model_test


✅ MLPipeline initialisé

📊 Configuration:
  Model name: consumption_model_test
  Model type: auto_gluon
  Test size: 0.25
  Random state: 42


## 2. Chargement des Données

Chargement des données depuis le fichier spécifié.

In [28]:
# Définir le chemin des données
DATA_PATH = pipeline.config.get('data', {}).get('train_path')
print(f'\n📂 Data path: {DATA_PATH}')

# Charger les données avec le pipeline
print('\n🔄 Chargement des données avec le pipeline...')
success = pipeline.step_1_load_data(DATA_PATH)

if success:
    print(f'\n✅ Données chargées: {pipeline.data.shape}')
    print(f'\nAperçu:')
    display(pipeline.data.head())
else:
    print('\n❌ Échec du chargement des données')

2026-06-08 00:58:57,936 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 1: CHARGEMENT DES DONNÉES ===TE



📂 Data path: ../data/test/train_consumption.parquet

🔄 Chargement des données avec le pipeline...
Données chargées avec succès: ..\data\test\train_consumption.parquet
Forme des données: (1488, 9)
Colonnes: ['Horodate', 'Valeur', 'temperature_2m_mean', 'relative_humidity_mean', 'precipitation_sum', 'is_vacances', 'nom_vacances', 'jour de la semaine', 'jour férié']
Types: {'Horodate': dtype('<M8[ns]'), 'Valeur': dtype('float64'), 'temperature_2m_mean': dtype('float64'), 'relative_humidity_mean': dtype('float64'), 'precipitation_sum': dtype('float64'), 'is_vacances': dtype('int64'), 'nom_vacances': dtype('O'), 'jour de la semaine': dtype('O'), 'jour férié': dtype('int64')}
Head:
             Horodate      Valeur  temperature_2m_mean  \
0 2024-01-01 00:00:00  524.835708             2.371718   
1 2024-01-01 00:30:00  493.086785            16.312921   
2 2024-01-01 01:00:00  482.384427             7.240709   
3 2024-01-01 01:30:00  526.151493            22.790996   
4 2024-01-01 02:00:00  3

,Horodate,Valeur,temperature_2m_mean,relative_humidity_mean,precipitation_sum,is_vacances,nom_vacances,jour de la semaine,jour férié
0,2024-01-01 00:00:00,524.835708,2.371718,82.637067,0.023052,1,Noël,Monday,1
1,2024-01-01 00:30:00,493.086785,16.312921,61.536840,0.043461,1,Noël,Monday,1
2,2024-01-01 01:00:00,482.384427,7.240709,75.434794,0.013480,1,Noël,Monday,1
3,2024-01-01 01:30:00,526.151493,22.790996,71.998104,0.013910,1,Noël,Monday,1
4,2024-01-01 02:00:00,388.292331,7.178762,72.640201,0.112762,1,Noël,Monday,1


## 3. Validation des Données

Validation de la qualité des données.

In [29]:
print('🔄 Validation des données...')
success = pipeline.step_2_validate_data(save_report=True)

if success:
    print('\n✅ Validation réussie')
else:
    print('\n⚠️ Validation échouée - Problèmes de qualité détectés')

2026-06-08 00:58:58,017 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 2: VALIDATION DES DONNÉES ===


🔄 Validation des données...


2026-06-08 00:58:58,031 - ml.utils.data.data_validator - INFO - Validation: Valeurs manquantes=0.00%, Doublons=0.00%
2026-06-08 00:58:58,036 - ml.utils.pipelines.training_pipeline - INFO - Résultats de validation: True
2026-06-08 00:58:58,040 - ml.utils.pipelines.training_pipeline - WARNING - Rapport Evidently non généré: 'report_path'



✅ Validation réussie


## 4. Transformation des Données

Nettoyage et transformation des données.

In [30]:
print('🔄 Transformation des données...')
success = pipeline.step_3_transform_data()

if success:
    print('\n✅ Transformation réussie')
    print(f'\nDonnées après transformation: {pipeline.data.shape}')
    display(pipeline.data.head())
else:
    print('\n❌ Échec de la transformation')

🔄 Transformation des données...

2026-06-08 00:58:58,089 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 3: TRANSFORMATION DES DONNÉES ===
2026-06-08 00:58:58,093 - ml.utils.data.data_transformer - WARNING - Colonnes à supprimer non trouvées dans les données: ['Identifiant PRM', 'Date de début', 'Date de fin', 'Grandeur physique', 'Grandeur métier', 'Etape métier', 'Unité', 'Nature', 'Pas', 'Indice de vraisemblance', 'Etat complémentaire']


2026-06-08 00:58:58,098 - ml.utils.data.data_transformer - INFO - Supprimé 1 colonne(s): ['Horodate']


2026-06-08 00:58:58,151 - ml.utils.data.data_transformer - INFO - Nettoyage des données terminé
2026-06-08 00:58:58,152 - ml.utils.pipelines.training_pipeline - INFO - ✓ Données nettoyées et transformées



✅ Transformation réussie

Données après transformation: (1488, 8)


,Valeur,temperature_2m_mean,relative_humidity_mean,precipitation_sum,is_vacances,nom_vacances,jour de la semaine,jour férié
0,524.835708,2.371718,82.637067,0.023052,1,Noël,Monday,1
1,493.086785,16.312921,61.536840,0.043461,1,Noël,Monday,1
2,482.384427,7.240709,75.434794,0.013480,1,Noël,Monday,1
3,526.151493,22.790996,71.998104,0.013910,1,Noël,Monday,1
4,388.292331,7.178762,72.640201,0.112762,1,Noël,Monday,1


print('🔄 Préparation des données...')
success = pipeline.step_3_prepare_data()

if success:
    print('\n✅ Préparation réussie')
    print(f'\n? Split:')
    print(f"  Train: {pipeline.X_train.shape}")
    print(f"  Test: {pipeline.X_test.shape}")
    print(f"\n📋 Features: {len(pipeline.feature_names)}")
    print(f"  {pipeline.feature_names}")
else:
    print('\n❌ Échec de la préparation')

In [31]:
print('🔄 Préparation des données...')
success = pipeline.step_3_prepare_data()

if success:
    print('\n✅ Préparation réussie')
    print(f'\n📊 Split:')
    print(f"  Train: {pipeline.X_train.shape}")
    print(f"  Test: {pipeline.X_test.shape}")
    print(f"\n📋 Features: {len(pipeline.feature_names)}")
    print(f"  {pipeline.feature_names}")
else:
    print('\n❌ Échec de la préparation')

2026-06-08 00:58:58,195 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 3: TRANSFORMATION DES DONNÉES ===


🔄 Préparation des données...
=== ÉTAPE 3: PRÉPARATION ET PRÉTRAITEMENT (stateless)===


2026-06-08 00:58:58,199 - ml.utils.data.data_transformer - WARNING - Colonnes à supprimer non trouvées dans les données: ['Identifiant PRM', 'Horodate', 'Date de début', 'Date de fin', 'Grandeur physique', 'Grandeur métier', 'Etape métier', 'Unité', 'Nature', 'Pas', 'Indice de vraisemblance', 'Etat complémentaire']
2026-06-08 00:58:58,201 - ml.utils.data.data_transformer - INFO - Aucune colonne valide à supprimer après vérification
2026-06-08 00:58:58,239 - ml.utils.data.data_transformer - INFO - Nettoyage des données terminé
2026-06-08 00:58:58,240 - ml.utils.pipelines.training_pipeline - INFO - ✓ Données nettoyées et transformées
2026-06-08 00:58:58,243 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 3: PRÉPARATION ET PRÉTRAITEMENT (stateful)===
2026-06-08 00:58:58,245 - ml.utils.pipelines.training_pipeline - INFO -   → Division train/test...
2026-06-08 00:58:58,246 - ml.utils.pipelines.training_pipeline - INFO -   i Colonne cible (config): Valeur
2026-06-08 00:58:58,249 - 


✅ Préparation réussie

📊 Split:
  Train: (1116, 7)
  Test: (372, 7)

📋 Features: 7
  ['temperature_2m_mean', 'relative_humidity_mean', 'precipitation_sum', 'is_vacances', 'nom_vacances', 'jour de la semaine', 'jour férié']


## 6. Entraînement du Modèle

Entraînement du modèle avec les données préparées.

In [32]:
print('🔄 Entraînement du modèle...')
print(f"  Type: {pipeline.config.get('model', {}).get('model_type')}")
success = pipeline.step_4_train_model()

if success:
    print('\n✅ Entraînement réussi')
    print(f'\n🤖 Modèle: {type(pipeline.model).__name__}')
else:
    print('\n❌ Échec de l\'entraînement')

2026-06-08 00:58:58,347 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 4: ENTRAÎNEMENT DU MODÈLE ===
2026-06-08 00:58:58,350 - ml.utils.models.model - INFO - X_train shape: (1116, 7), y_train shape: (1116,)
No path specified. Models will be saved in: "AutogluonModels\ag-20260607_225858"


🔄 Entraînement du modèle...
  Type: auto_gluon


Preset alias specified: 'good' maps to 'good_quality'.
Verbosity: 2 (Standard Logging)
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          4
Pytorch Version:    Can't import torch
CUDA Version:       Can't get cuda version from torch
Memory Avail:       3.31 GB / 15.88 GB (20.8%)
Disk Space Avail:   21.87 GB / 473.53 GB (4.6%)
Presets specified: ['good']
Using hyperparameters preset: hyperparameters='light'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=0, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "c:\_sources\Jenedai\jinsudai\notebooks\AutogluonModels\ag-20260607_225858"
Train Data Rows:    1116
Train Da


✅ Entraînement réussi

🤖 Modèle: TabularPredictor


## 7. Évaluation du Modèle

Évaluation du modèle sur les données de test.

In [33]:
print('🔄 Évaluation du modèle...')
success = pipeline.step_5_evaluate_model()

if success:
    print('\n✅ Évaluation réussie')
    print(f'\n📈 Métriques:')
    for metric, value in pipeline.metrics.items():
        print(f"  {metric}: {value:.4f}")
else:
    print('\n❌ Échec de l\'évaluation')

2026-06-08 00:59:12,300 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 5: ÉVALUATION DU MODÈLE ===


🔄 Évaluation du modèle...


2026-06-08 00:59:13,875 - ml.utils.models.model - WARNING - Calcul manuel des métriques AutoGluon a échoué: continuous is not supported
2026-06-08 00:59:13,876 - ml.utils.models.model - INFO - Autogluon - Métriques: {'root_mean_squared_error': np.float64(-227.80280852140217), 'mean_squared_error': -51894.11957023862, 'mean_absolute_error': -195.97060364684387, 'r2': -0.046672590498818556, 'pearsonr': 0.05037216860885269, 'median_absolute_error': -189.83427212083626}



✅ Évaluation réussie

📈 Métriques:
  root_mean_squared_error: -227.8028
  mean_squared_error: -51894.1196
  mean_absolute_error: -195.9706
  r2: -0.0467
  pearsonr: 0.0504
  median_absolute_error: -189.8343


## 8. Monitoring

In [34]:
print('🔄 Monitoring de la performance...')
success = pipeline.step_6_monitor_performance()

if success:
    print('\n✅ Monitoring réussi')
    print(f'\n📊 Résultats du monitoring:')
    if hasattr(pipeline, 'monitoring_results'):
        drift = pipeline.monitoring_results.get('drift', {})
        print(f"  Drift détecté: {drift.get('drift_detected', False)}")
        print(f"  Mean drift: {drift.get('mean_drift', 0):.4f}")
        
        perf_test = pipeline.monitoring_results.get('performance_test', {})
        if perf_test:
            print(f'\n  Performance Test:')
            for metric, value in perf_test.items():
                print(f"    {metric}: {value:.4f}")
        
        perf_train = pipeline.monitoring_results.get('performance_train', {})
        if perf_train:
            print(f'\n  Performance Train:')
            for metric, value in perf_train.items():
                print(f"    {metric}: {value:.4f}")
else:
    print('\n❌ Échec du monitoring')

2026-06-08 00:59:13,909 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 6: MONITORING DE LA PERFORMANCE ===


🔄 Monitoring de la performance...


2026-06-08 00:59:15,248 - ml.utils.monitoring.performance_monitor - WARNING - Drift détecté! Mean drift: 0.0057, Std drift: 0.4788
2026-06-08 00:59:15,250 - ml.utils.monitoring.performance_monitor - WARNING - Échec des métriques de classification: continuous is not supported, tentative avec régression
2026-06-08 00:59:15,258 - ml.utils.monitoring.performance_monitor - INFO - Métriques de performance calculées: {'mae': 106.3124703569389, 'mse': 14865.652032019561, 'rmse': np.float64(121.92478022132974), 'r2': 0.7011418378409362}
2026-06-08 00:59:15,261 - ml.utils.monitoring.performance_monitor - WARNING - Échec des métriques de classification: continuous is not supported, tentative avec régression
2026-06-08 00:59:15,267 - ml.utils.monitoring.performance_monitor - INFO - Métriques de performance calculées: {'mae': 195.97060364684387, 'mse': 51894.11957023862, 'rmse': np.float64(227.80280852140217), 'r2': -0.046672590498818556}
2026-06-08 00:59:15,276 - ml.utils.pipelines.training_pipeli


✅ Monitoring réussi

📊 Résultats du monitoring:
  Drift détecté: True
  Mean drift: 0.0057

  Performance Test:
    mae: 195.9706
    mse: 51894.1196
    rmse: 227.8028
    r2: -0.0467

  Performance Train:
    mae: 106.3125
    mse: 14865.6520
    rmse: 121.9248
    r2: 0.7011


## 9. Logging MLflow

Enregistrement des métriques et du modèle dans MLflow.

In [35]:
print('🔄 Logging avec MLflow...')
print(f"  Tracking URI: {pipeline.config.get('mlflow', {}).get('tracking_uri')}")
print(f"  Experiment: {pipeline.config.get('mlflow', {}).get('experiment_name')}")
success = pipeline.step_7_log_with_mlflow()

if success:
    print('\n✅ Logging MLflow réussi')
else:
    print('\n❌ Échec du logging MLflow')

2026-06-08 00:59:15,317 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 7: LOGGING AVEC MLFLOW ===
2026-06-08 00:59:15,321 - ml.utils.models.mlflow_tracker - INFO - Tracking URI configuré: https://jinsudai-mlflow.hf.space/
2026-06-08 00:59:15,326 - ml.utils.models.mlflow_tracker - INFO - Répertoire local des artefacts configuré: C:\_sources\Jenedai\jinsudai\src\jinsudai\mlflow


🔄 Logging avec MLflow...
  Tracking URI: https://jinsudai-mlflow.hf.space/
  Experiment: energy_consumption_test


2026-06-08 00:59:17,257 - ml.utils.models.mlflow_tracker - INFO - Run MLflow démarrée - Expérience: energy_consumption_test
2026-06-08 00:59:18,657 - ml.utils.models.mlflow_tracker - INFO - Paramètres enregistrés: {'test_size': 0.25, 'random_state': 42, 'model_type': 'auto_gluon', 'problem_type': 'regression', 'time_limit': 1800}
2026-06-08 00:59:32,289 - ml.utils.models.mlflow_tracker - INFO - Métriques enregistrées: ['root_mean_squared_error', 'mean_squared_error', 'mean_absolute_error', 'r2', 'pearsonr', 'median_absolute_error', 'monitoring_drift_mean_drift', 'monitoring_drift_std_drift', 'monitoring_drift_drift_detected', 'monitoring_test_mae', 'monitoring_test_mse', 'monitoring_test_rmse', 'monitoring_test_r2', 'monitoring_train_mae', 'monitoring_train_mse', 'monitoring_train_rmse', 'monitoring_train_r2']
2026-06-08 00:59:32,291 - ml.utils.models.mlflow_tracker - INFO - Chemin AutoGluon détecté: c:\_sources\Jenedai\jinsudai\notebooks\AutogluonModels\ag-20260607_225858
2026-06-08 0

🏃 View run dashing-mole-862 at: https://jinsudai-mlflow.hf.space/#/experiments/2/runs/ad61f0533616464ca658f27c36285af9
🧪 View experiment at: https://jinsudai-mlflow.hf.space/#/experiments/2


2026-06-08 01:00:38,497 - ml.utils.models.mlflow_tracker - INFO - Run MLflow terminée
2026-06-08 01:00:38,500 - ml.utils.models.mlflow_tracker - INFO - Session d'entraînement enregistrée dans MLflow



✅ Logging MLflow réussi


## 10. Gestion des Stages MLflow

Enregistrement du modèle dans le Model Registry et promotion vers Production.

In [36]:
print('🔄 Gestion des stages MLflow...')
print(f"  Model name: {pipeline.model_name}")

# Définir les métriques pour la comparaison (pour la consommation)
metric_keys = ["mae", "rmse", "r2"]
min_improvement = 0.0  # 0% = promouvoir même sans amélioration

success = pipeline.step_9_manage_model_stages(
    metric_keys=metric_keys,
    min_improvement=min_improvement
)

if success:
    print('\n✅ Gestion des stages réussie')
    if pipeline.promotion_result:
        print(f'\n📊 Résultat de la promotion:')
        print(f"  Success: {pipeline.promotion_result['success']}")
        print(f"  Version: {pipeline.promotion_result.get('version', 'N/A')}")
        print(f"  Reason: {pipeline.promotion_result.get('reason', 'N/A')}")
else:
    print('\n❌ Échec de la gestion des stages')

2026-06-08 01:00:38,538 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 8: GESTION DES ALIASES DU MODÈLE ===
2026-06-08 01:00:38,542 - ml.utils.models.mlflow_tracker - INFO - Tracking URI configuré: https://jinsudai-mlflow.hf.space/
2026-06-08 01:00:38,544 - ml.utils.models.mlflow_tracker - INFO - Répertoire local des artefacts configuré: C:\_sources\Jenedai\jinsudai\src\jinsudai\mlflow


🔄 Gestion des stages MLflow...
  Model name: consumption_model_test


2026-06-08 01:00:39,236 - ml.utils.pipelines.training_pipeline - INFO - 
[1/4] Récupération du run_id et enregistrement du modèle en Staging...
2026-06-08 01:00:39,237 - ml.utils.pipelines.training_pipeline - INFO -   Métriques à comparer (priorité): ['mae', 'rmse', 'r2']
2026-06-08 01:00:40,923 - ml.utils.pipelines.training_pipeline - INFO -   ℹ Run trouvée: ad61f0533616464ca658f27c36285af9
2026-06-08 01:00:42,006 - ml.utils.models.mlflow_tracker - INFO - Registered Model 'consumption_model_test' existe déjà
2026/06/08 01:00:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: consumption_model_test, version 3
2026-06-08 01:00:43,317 - ml.utils.models.mlflow_tracker - INFO - Modèle consumption_model_test v3 enregistré dans le registry
2026-06-08 01:00:43,318 - ml.utils.pipelines.training_pipeline - INFO -   ✓ Modèle consumption_model_test v3 enregistré
2026-06-08 01:00:43,319 - ml.utils.pipelines.training_pipel


✅ Gestion des stages réussie

📊 Résultat de la promotion:
  Success: False
  Version: N/A
  Reason: model_not_better


## 11. Pipeline Complet (One-shot)

Exécution de toutes les étapes en une seule fois.

In [37]:
# Réinitialiser le pipeline
pipeline = MLPipeline()

# Exécuter le pipeline complet
print('🚀 Exécution du pipeline complet...\n')
success = pipeline.run_full_pipeline(DATA_PATH)

if success:
    print('\n🎉 Pipeline terminé avec succès!')
    print(f'\n📊 Résultats finaux:')
    print(f"  Model: {type(pipeline.model).__name__}")
    print(f"  Métriques: {pipeline.metrics}")
    if pipeline.version_staging:
        print(f"  Version enregistrée: {pipeline.version_staging}")
else:
    print('\n❌ Pipeline échoué')

2026-06-08 01:00:48,160 - ml.utils.pipelines.training_pipeline - INFO - Pipeline MLOps initialisé avec model_name=consumption_model_test
2026-06-08 01:00:48,165 - ml.utils.pipelines.training_pipeline - INFO - 
2026-06-08 01:00:48,169 - ml.utils.pipelines.training_pipeline - INFO - DÉMARRAGE DU PIPELINE COMPLET
2026-06-08 01:00:48,172 - ml.utils.pipelines.training_pipeline - INFO - ==================================================

2026-06-08 01:00:48,173 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 1: CHARGEMENT DES DONNÉES ===TE
2026-06-08 01:00:48,204 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 2: VALIDATION DES DONNÉES ===
2026-06-08 01:00:48,218 - ml.utils.data.data_validator - INFO - Validation: Valeurs manquantes=0.00%, Doublons=0.00%
2026-06-08 01:00:48,221 - ml.utils.pipelines.training_pipeline - INFO - Résultats de validation: True
2026-06-08 01:00:48,223 - ml.utils.pipelines.training_pipeline - WARNING - Rapport Evidently non généré: 'report_path'


🚀 Exécution du pipeline complet...

Données chargées avec succès: ..\data\test\train_consumption.parquet
Forme des données: (1488, 9)
Colonnes: ['Horodate', 'Valeur', 'temperature_2m_mean', 'relative_humidity_mean', 'precipitation_sum', 'is_vacances', 'nom_vacances', 'jour de la semaine', 'jour férié']
Types: {'Horodate': dtype('<M8[ns]'), 'Valeur': dtype('float64'), 'temperature_2m_mean': dtype('float64'), 'relative_humidity_mean': dtype('float64'), 'precipitation_sum': dtype('float64'), 'is_vacances': dtype('int64'), 'nom_vacances': dtype('O'), 'jour de la semaine': dtype('O'), 'jour férié': dtype('int64')}
Head:
             Horodate      Valeur  temperature_2m_mean  \
0 2024-01-01 00:00:00  524.835708             2.371718   
1 2024-01-01 00:30:00  493.086785            16.312921   
2 2024-01-01 01:00:00  482.384427             7.240709   
3 2024-01-01 01:30:00  526.151493            22.790996   
4 2024-01-01 02:00:00  388.292331             7.178762   

   relative_humidity_mean  p

2026-06-08 01:00:48,283 - ml.utils.data.data_transformer - INFO - Nettoyage des données terminé
2026-06-08 01:00:48,286 - ml.utils.pipelines.training_pipeline - INFO - ✓ Données nettoyées et transformées
2026-06-08 01:00:48,288 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 3: TRANSFORMATION DES DONNÉES ===
2026-06-08 01:00:48,291 - ml.utils.data.data_transformer - WARNING - Colonnes à supprimer non trouvées dans les données: ['Identifiant PRM', 'Horodate', 'Date de début', 'Date de fin', 'Grandeur physique', 'Grandeur métier', 'Etape métier', 'Unité', 'Nature', 'Pas', 'Indice de vraisemblance', 'Etat complémentaire']
2026-06-08 01:00:48,293 - ml.utils.data.data_transformer - INFO - Aucune colonne valide à supprimer après vérification


=== ÉTAPE 3: PRÉPARATION ET PRÉTRAITEMENT (stateless)===


2026-06-08 01:00:48,338 - ml.utils.data.data_transformer - INFO - Nettoyage des données terminé
2026-06-08 01:00:48,341 - ml.utils.pipelines.training_pipeline - INFO - ✓ Données nettoyées et transformées
2026-06-08 01:00:48,342 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 3: PRÉPARATION ET PRÉTRAITEMENT (stateful)===
2026-06-08 01:00:48,347 - ml.utils.pipelines.training_pipeline - INFO -   → Division train/test...
2026-06-08 01:00:48,348 - ml.utils.pipelines.training_pipeline - INFO -   i Colonne cible (config): Valeur
2026-06-08 01:00:48,351 - ml.utils.data.data_preparation - INFO - Données après nettoyage: (1488, 8)
2026-06-08 01:00:48,353 - ml.utils.data.data_preparation - INFO - Aperçu des premières lignes de data_clean:
2026-06-08 01:00:48,358 - ml.utils.data.data_preparation - INFO -        Valeur  temperature_2m_mean  relative_humidity_mean  precipitation_sum  \
0  524.835708             2.371718               82.637067           0.023052   
1  493.086785           

🏃 View run capricious-quail-573 at: https://jinsudai-mlflow.hf.space/#/experiments/2/runs/718665e959974c8fa080524c450d8af2
🧪 View experiment at: https://jinsudai-mlflow.hf.space/#/experiments/2


2026-06-08 01:02:17,211 - ml.utils.models.mlflow_tracker - INFO - Run MLflow terminée
2026-06-08 01:02:17,212 - ml.utils.models.mlflow_tracker - INFO - Session d'entraînement enregistrée dans MLflow
2026-06-08 01:02:17,214 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 8: NETTOYAGE DU MODÈLE LOCAL ===
2026-06-08 01:02:17,274 - ml.utils.pipelines.training_pipeline - INFO - Modèle AutoGluon original supprimé: c:\_sources\Jenedai\jinsudai\notebooks\AutogluonModels\ag-20260607_230048
2026-06-08 01:02:17,276 - ml.utils.pipelines.training_pipeline - INFO - === ÉTAPE 8: GESTION DES ALIASES DU MODÈLE ===
2026-06-08 01:02:17,279 - ml.utils.models.mlflow_tracker - INFO - Tracking URI configuré: https://jinsudai-mlflow.hf.space/
2026-06-08 01:02:17,281 - ml.utils.models.mlflow_tracker - INFO - Répertoire local des artefacts configuré: C:\_sources\Jenedai\jinsudai\src\jinsudai\mlflow
2026-06-08 01:02:17,908 - ml.utils.pipelines.training_pipeline - INFO - 
[1/4] Récupération du run_id et


🎉 Pipeline terminé avec succès!

📊 Résultats finaux:
  Model: TabularPredictor
  Métriques: {'root_mean_squared_error': np.float64(-227.80280852140217), 'mean_squared_error': -51894.11957023862, 'mean_absolute_error': -195.97060364684387, 'r2': -0.046672590498818556, 'pearsonr': 0.05037216860885269, 'median_absolute_error': -189.83427212083626}
  Version enregistrée: 4


## Résumé

Ce notebook permet d'exécuter manuellement chaque étape du pipeline d'entraînement en utilisant la classe `MLPipeline` :

- **Étape 1** : Chargement des données avec `step_1_load_data`
- **Étape 2** : Validation des données avec `step_2_validate_data`
- **Étape 3** : Transformation des données avec `step_3_transform_data`
- **Étape 4** : Préparation des données avec `step_3_prepare_data`
- **Étape 5** : Entraînement du modèle avec `step_4_train_model`
- **Étape 6** : Évaluation du modèle avec `step_5_evaluate_model`
- **Étape 7** : Monitoring avec `step_6_monitor_performance`
- **Étape 8** : Logging MLflow avec `step_7_log_with_mlflow`
- **Étape 9** : Gestion des stages avec `step_8_manage_model_stages`
- **Étape 10** : Pipeline complet avec `run_full_pipeline`

### Attributs disponibles après exécution :
- `pipeline.config` : Configuration chargée
- `pipeline.data` : Données brutes
- `pipeline.X_train, pipeline.X_test` : Features splitées
- `pipeline.y_train, pipeline.y_test` : Targets splités
- `pipeline.model` : Modèle entraîné
- `pipeline.metrics` : Métriques d'évaluation
- `pipeline.monitoring_results` : Résultats du monitoring
- `pipeline.version_staging` : Version du modèle enregistrée
- `pipeline.promotion_result` : Résultat de la promotion